# HubbardBath 08 — Thermal polynomials and QSVT readiness

Milestone 07 gave us a block encoding of $X=H/\alpha$. This milestone asks what polynomial transformation of $X$ would produce the thermal amplitude underlying a Gibbs state.

The important subtlety is **boundedness**: a QSVT target must be bounded on the encoded interval $[-1,1]$.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial.chebyshev import chebval

candidates=[Path.cwd(),Path.cwd().parent,Path.cwd()/'HubbardBath']
for c in candidates:
    if (c/'src'/'thermal_polynomial.py').exists():
        repo_root=c.resolve(); break
else:
    raise FileNotFoundError('Run from the HubbardBath repository')
if str(repo_root) not in sys.path:
    sys.path.insert(0,str(repo_root))

from src.hubbard import two_site_hubbard_hamiltonian
from src.qubit_mapping import (
    lcu_normalization, remove_zero_terms,
    two_site_hubbard_pauli_terms,
)
from src.thermal_polynomial import (
    amplitude_operator_error, bounded_thermal_function,
    gibbs_trace_distance_error, minimum_degree_for_error,
    parity_components, qsvt_readiness_report,
)


## 1. Why we shift the thermal function

The block encoding represents $X=H/\alpha$. Instead of approximating $e^{-\beta H/2}$ directly, we use

$$f(x)=\exp[-\beta\alpha(x+1)/2].$$

This obeys $0<f(x)\leq1$ for all $x\in[-1,1]$. At the matrix level,

$$f(H/\alpha)=e^{-\beta\alpha/2}e^{-\beta H/2}.$$

The scalar factor disappears when the squared amplitude is normalized into a density matrix.

In [ ]:
U=4.0
beta=1.0
H=two_site_hubbard_hamiltonian(t=1.0,U=U,mu=0.0)
terms=remove_zero_terms(two_site_hubbard_pauli_terms(t=1.0,U=U,mu=0.0))
alpha=lcu_normalization(terms)
print('LCU alpha =',alpha)

x=np.linspace(-1,1,1001)
f=bounded_thermal_function(x,beta,alpha)
print('Thermal target range:',f.min(),f.max())


## 2. Find the polynomial degree needed for a target error

We use Chebyshev interpolation on the full interval $[-1,1]$ and search for the smallest degree satisfying a chosen uniform scalar error.

In [ ]:
for epsilon in [1e-2,1e-4,1e-6]:
    approx=minimum_degree_for_error(beta,alpha,epsilon,max_degree=256)
    print(f'epsilon={epsilon:.0e}: degree={approx.degree}, actual error={approx.max_scalar_error:.3e}')


In [ ]:
approx=minimum_degree_for_error(beta,alpha,1e-4,max_degree=256)
p=chebval(x,approx.coefficients)
plt.figure(figsize=(7,4.8))
plt.plot(x,f,label='bounded thermal target')
plt.plot(x,p,linestyle='--',label=f'Chebyshev degree {approx.degree}')
plt.xlabel(r'Encoded eigenvalue $x$')
plt.ylabel('Thermal amplitude')
plt.title(r'Approximation of $e^{-\beta\alpha(x+1)/2}$')
plt.legend(); plt.grid(alpha=0.2); plt.tight_layout(); plt.show()


## 3. Verify the matrix transformation, not just the scalar curve

The scalar approximation is only useful if $p(H/\alpha)$ approximates the intended thermal matrix function. We evaluate the Chebyshev series directly on the Hamiltonian and compare against exact diagonalization.

In [ ]:
print(f'Amplitude operator error: {amplitude_operator_error(H,approx):.3e}')
print(f'Gibbs-state trace distance: {gibbs_trace_distance_error(H,approx):.3e}')


## 4. QSVT parity constraint

A standard single QSP/QSVT polynomial sequence has a fixed parity. Our thermal target is neither purely even nor purely odd, so we split the Chebyshev series into parity-compatible pieces.

In [ ]:
even,odd=parity_components(approx.coefficients)
report=qsvt_readiness_report(approx)
report


In [ ]:
plt.figure(figsize=(7,4.8))
plt.plot(x,chebval(x,even),label='even component')
plt.plot(x,chebval(x,odd),label='odd component')
plt.plot(x,chebval(x,even)+chebval(x,odd),linestyle='--',label='sum')
plt.xlabel(r'$x$'); plt.ylabel('Polynomial value')
plt.title('Parity decomposition for QSVT-oriented synthesis')
plt.legend(); plt.grid(alpha=0.2); plt.tight_layout(); plt.show()


## 5. Temperature and interaction sweep

The algorithmic difficulty proxy is the minimum polynomial degree needed at fixed error. Because the target depends on $\beta\alpha$, colder temperatures and larger LCU normalization generally demand higher degree.

In [ ]:
U_values=[0.0,2.0,4.0,6.0,8.0]
beta_values=[0.25,0.5,1.0,2.0]
rows=[]
for U_value in U_values:
    terms_u=remove_zero_terms(two_site_hubbard_pauli_terms(t=1.0,U=U_value,mu=0.0))
    alpha_u=lcu_normalization(terms_u)
    for beta_value in beta_values:
        a=minimum_degree_for_error(beta_value,alpha_u,1e-4,max_degree=256)
        rows.append((U_value,beta_value,alpha_u,beta_value*alpha_u,a.degree))

for row in rows:
    print('U/t=%3.1f beta*t=%4.2f alpha=%5.2f beta*alpha=%5.2f degree=%d'%row)


## What we can honestly claim after this milestone

We have a **bounded, parity-decomposed polynomial transformation** of the block-encoded Hubbard Hamiltonian and verify that it reproduces the Gibbs state after normalization.

We do **not** yet claim that QSP phase angles have been synthesized or that a fault-tolerant QSVT circuit has been compiled. Those are separate implementation steps.

The final comparison will place this polynomial-degree measure beside the physical Liouvillian gap and mixing time measured earlier.